In [1]:
import json, random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms as T
from torchvision.transforms import InterpolationMode

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cpu")

In [2]:
mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

eval_transform = T.Compose([
    T.Resize((200, 200), interpolation=InterpolationMode.BILINEAR),
    T.ToTensor(), T.Normalize(mean, std),
])

aug_transform = T.Compose([
    T.Resize((200, 200), interpolation=InterpolationMode.BILINEAR),
    T.RandomRotation(50, interpolation=InterpolationMode.NEAREST),
    T.RandomResizedCrop(200, scale=(0.9, 1.0), ratio=(0.9, 1.1),
                        interpolation=InterpolationMode.BILINEAR),
    T.RandomHorizontalFlip(),
    T.ToTensor(), T.Normalize(mean, std),
])

train_dataset = datasets.ImageFolder("data/train", transform=eval_transform)
train_dataset_aug = datasets.ImageFolder("data/train", transform=aug_transform)
evaluation_dataset = datasets.ImageFolder("data/test", transform=eval_transform)

print(len(train_dataset), len(evaluation_dataset), train_dataset.class_to_idx)

800 201 {'curly': 0, 'straight': 1}


In [13]:
loader_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    train_dataset,
    batch_size=20,
    shuffle=True,
    num_workers=0,
    generator=loader_generator,
)
train_aug_loader = DataLoader(
    train_dataset_aug,
    batch_size=20,
    shuffle=True,
    num_workers=0,
    generator=loader_generator,
)
evaluation_loader = DataLoader(
    evaluation_dataset,
    batch_size=20,
    shuffle=False,
    num_workers=0,
)

In [4]:
class HairModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=0)
        self.pool = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(32 * 99 * 99, 64)
        self.fc2 = nn.Linear(64, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.pool(self.relu(self.conv(x)))
        x = x.flatten(1)
        x = self.relu(self.fc1(x))
        return self.fc2(x)

model = HairModel().to(device)
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.002, momentum=0.8)

In [5]:
def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(training):
        for x, y in loader:
            x, y = x.to(device), y.float().unsqueeze(1).to(device)
            logits = model(x)
            loss = loss_fn(logits, y)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * x.size(0)
            correct += ((torch.sigmoid(logits) >= 0.5).float() == y).sum().item()
            n += x.size(0)
    return total_loss / n, correct / n

def fit(epochs, loader):
    h = {"train_loss": [], "train_acc": [], "eval_loss": [], "eval_acc": []}
    for epoch in range(epochs):
        tl, ta = run_epoch(model, loader, optimizer)
        el, ea = run_epoch(model, evaluation_loader)
        for k, v in zip(h, (tl, ta, el, ea)):
            h[k].append(v)
        print(f"{epoch+1:2d} | train {tl:.4f}/{ta:.4f} | eval {el:.4f}/{ea:.4f}")
    return h

In [6]:
history_baseline = fit(10, train_loader)

 1 | train 0.6385/0.6450 | eval 0.6010/0.6567
 2 | train 0.5441/0.7175 | eval 0.6003/0.6517
 3 | train 0.4955/0.7550 | eval 0.6405/0.6219
 4 | train 0.4330/0.7913 | eval 0.6274/0.6915
 5 | train 0.4048/0.8087 | eval 0.6293/0.6617
 6 | train 0.3953/0.8075 | eval 0.6332/0.7114
 7 | train 0.3437/0.8363 | eval 0.6895/0.6716
 8 | train 0.2806/0.8850 | eval 0.6497/0.7214
 9 | train 0.2236/0.9100 | eval 0.7114/0.7164
10 | train 0.1584/0.9387 | eval 1.0212/0.6368


In [7]:
from pathlib import Path

Path("runs/my").mkdir(parents=True, exist_ok=True)

with open("runs/my/history_baseline.json", "w") as f:
    json.dump(history_baseline, f, indent=2)

Question 2

In [8]:
sum(p.numel() for p in model.parameters())

20073473

Question 3

In [11]:
np.median(history_baseline["train_acc"]).round(2)

np.float64(0.81)

Question 4

In [12]:
np.std(history_baseline["train_loss"]).round(3)

np.float64(0.139)

Question 5

In [14]:
history_augmented = fit(10, train_aug_loader)

 1 | train 0.6520/0.6288 | eval 0.6635/0.7065
 2 | train 0.6077/0.6663 | eval 0.5996/0.6567
 3 | train 0.5767/0.6900 | eval 0.6416/0.7114
 4 | train 0.5559/0.6987 | eval 0.6268/0.6766
 5 | train 0.5477/0.7075 | eval 0.5714/0.7114
 6 | train 0.5418/0.7188 | eval 0.5652/0.7413
 7 | train 0.5128/0.7325 | eval 0.5675/0.7363
 8 | train 0.4957/0.7512 | eval 0.5558/0.7114
 9 | train 0.4819/0.7488 | eval 0.5377/0.7711
10 | train 0.4833/0.7588 | eval 0.5790/0.6965


In [15]:
with open("runs/my/history_augmented.json", "w") as f:
    json.dump(history_augmented, f, indent=2)

In [21]:
np.mean(history_augmented["eval_loss"]).round(3)

np.float64(0.591)

Question 6

In [28]:
np.mean(history_augmented["eval_acc"][-5:]).round(2)

np.float64(0.73)